# Customer Support Intelligence — One Complete End-to-End NLP Project

**Goal:** learn how an NLP system is designed, validated, trained, evaluated, packaged and monitored — not merely how to call a model.

This notebook is a single independently executable project. It covers the complete path:

business problem → educational raw data → data contract → EDA → leakage control → cleaning → train/validation/test → baselines → TF-IDF → model selection → final test → error analysis → entity extraction → retrieval → serialization → inference → robustness → monitoring → retraining.

### Reproducibility contract

- random seed: **42**
- no remote API calls
- no hidden dataset download
- raw educational data is generated deterministically by this notebook
- preprocessing is fitted inside scikit-learn pipelines
- validation chooses the model/hyperparameters
- the test set is untouched until final evaluation
- results and artifacts are written under `artifacts/`
- the serialized model is reloaded and tested before the notebook ends

The dataset is synthetic/curated for education. Its scores demonstrate engineering process, not real-world production quality.

## 0. Environment and project layout

Use the repository Conda environment:

    conda env create -f nlp/environment.yml
    conda activate awesome-nlp
    python -m ipykernel install --user --name awesome-nlp --display-name "Python (awesome-nlp)"
    jupyter lab

The path resolver lets the notebook run either from this project directory or from the repository root.

In [1]:
from pathlib import Path
import json, math, random, re, sys, platform, warnings, unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import __version__ as sklearn_version
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from joblib import dump, load

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def locate_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "nlp/projects/customer_support_intelligence",
        cwd.parent,
        cwd.parent.parent,
        cwd.parent.parent.parent,
    ]
    for candidate in candidates:
        if candidate.name == "customer_support_intelligence":
            return candidate
        if (candidate / "customer_support_intelligence_end_to_end.ipynb").exists():
            return candidate
        nested = candidate / "nlp/projects/customer_support_intelligence"
        if nested.exists():
            return nested
    raise FileNotFoundError(
        "Could not locate nlp/projects/customer_support_intelligence. "
        "Run from the project folder or repository root."
    )

PROJECT_ROOT = locate_project_root()
DATA_RAW = PROJECT_ROOT / "data/raw"
DATA_INTERIM = PROJECT_ROOT / "data/interim"
DATA_PROCESSED = PROJECT_ROOT / "data/processed"
ARTIFACTS = PROJECT_ROOT / "artifacts"

for p in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, ARTIFACTS]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root resolved successfully")
print("Python:", platform.python_version())
print("scikit-learn:", sklearn_version)
print("Random seed:", RANDOM_SEED)

Project root resolved successfully
Python: 3.x
scikit-learn: 1.x
Random seed: 42


## 1. Business problem → ML problem → system contract

### Business objective
Automatically triage support tickets and surface the most relevant support guidance while preserving a safe path to human review.

### NLP components
1. **Intent classification:** billing, refund, access, technical, account, delivery.
2. **Entity extraction:** amount, order ID, email and event date.
3. **Knowledge retrieval:** rank support articles.
4. **Confidence / abstention:** low-confidence cases can be escalated.

### Metrics
- intent classification → **macro-F1**
- entity extraction → exact-value precision / recall / F1
- retrieval → **Recall@1, Recall@3, MRR**
- deployment → coverage, accepted-case quality, drift indicators

Input example: `{"text": "My card was charged twice for ₹4,299"}`

Output contract includes intent, confidence, extracted entities, recommended article and human-review flag.

## 2. Build the educational raw dataset deterministically

There is **no hidden download step**. The notebook itself constructs the educational dataset and writes normal CSV files to `data/raw/`.

The generated corpus deliberately includes:
- six intents with modest class imbalance,
- typos, casing and punctuation noise,
- controlled multi-intent messages,
- amounts, order IDs, emails and dates,
- exact duplicates to demonstrate leakage prevention,
- a 12-article knowledge base with retrieval relevance ground truth.

In [2]:
def build_educational_dataset(seed=42):
    rng = random.Random(seed)

    kb_rows = [
        ("KB-BILL-01","billing","Duplicate card payment","A card transaction appears more than once. Check whether one entry is pending; if both settle, raise a duplicate-charge dispute."),
        ("KB-BILL-02","billing","Unexpected fee or incorrect charge","Review fees, merchant descriptors and invoice amounts. If the amount is wrong or a fee is unexplained, open a billing investigation."),
        ("KB-REF-01","refund","Refund pending","Refunds may take several business days after a merchant releases them. Verify the refund reference and expected settlement window."),
        ("KB-REF-02","refund","Refund amount incorrect","If a refund arrived for the wrong amount, compare the original transaction, refund reference and merchant confirmation."),
        ("KB-ACC-01","access","Password and account lock","Reset a forgotten password using the recovery flow. Locked accounts may require identity verification before access is restored."),
        ("KB-ACC-02","access","OTP or authentication problem","For missing or invalid OTP codes, verify the registered contact method, device time and authenticator setup."),
        ("KB-TECH-01","technical","Application crash or freeze","For crashes and freezes, capture app version, device details and reproduction steps, then retry after cache/update checks."),
        ("KB-TECH-02","technical","Network, sync or upload error","For timeouts, sync failures or upload errors, verify connectivity, service status and file constraints before escalation."),
        ("KB-ACCT-01","account","Update profile details","Customers can update email, phone, address and profile preferences after identity verification."),
        ("KB-ACCT-02","account","Close or manage account","Account closure and preference changes require confirmation and may have retention or balance prerequisites."),
        ("KB-DEL-01","delivery","Order delayed or missing","Track the shipment using the order ID. Delayed or missing deliveries may require carrier investigation."),
        ("KB-DEL-02","delivery","Wrong or damaged delivery","If an item is wrong or damaged, capture the order ID, item details and evidence before replacement or return."),
    ]
    kb = pd.DataFrame(kb_rows, columns=["article_id","intent","title","content"])

    templates = {
        "billing": [
            ("KB-BILL-01","my card was charged twice for the same purchase"),
            ("KB-BILL-01","I see a duplicate transaction on my card"),
            ("KB-BILL-01","the same merchant payment appears two times"),
            ("KB-BILL-02","why was I charged an extra service fee"),
            ("KB-BILL-02","the amount on my bill looks incorrect"),
            ("KB-BILL-02","I do not recognize this merchant charge"),
        ],
        "refund": [
            ("KB-REF-01","my refund is still missing"),
            ("KB-REF-01","the merchant says refunded but I cannot see it"),
            ("KB-REF-01","when will the refund appear on my statement"),
            ("KB-REF-02","the refund amount is lower than expected"),
            ("KB-REF-02","I received only part of my refund"),
            ("KB-REF-02","the refunded amount does not match the purchase"),
        ],
        "access": [
            ("KB-ACC-01","I forgot my password and cannot sign in"),
            ("KB-ACC-01","my account is locked after too many login attempts"),
            ("KB-ACC-01","password reset is not letting me access the account"),
            ("KB-ACC-02","the OTP code never arrives"),
            ("KB-ACC-02","my verification code keeps showing invalid"),
            ("KB-ACC-02","two factor authentication stopped working"),
        ],
        "technical": [
            ("KB-TECH-01","the mobile app crashes whenever I open it"),
            ("KB-TECH-01","the screen freezes after the latest update"),
            ("KB-TECH-01","the application closes when I tap settings"),
            ("KB-TECH-02","file upload fails with a network error"),
            ("KB-TECH-02","data synchronization is not working"),
            ("KB-TECH-02","the dashboard keeps timing out"),
        ],
        "account": [
            ("KB-ACCT-01","please change the email on my profile"),
            ("KB-ACCT-01","I need to update my phone number"),
            ("KB-ACCT-01","how can I change my mailing address"),
            ("KB-ACCT-02","I want to close my account permanently"),
            ("KB-ACCT-02","change my communication preferences"),
            ("KB-ACCT-02","I want to manage my account settings"),
        ],
        "delivery": [
            ("KB-DEL-01","my package has not arrived yet"),
            ("KB-DEL-01","the delivery is late and tracking has not moved"),
            ("KB-DEL-01","where is my order it should be here already"),
            ("KB-DEL-02","I received the wrong product"),
            ("KB-DEL-02","the item arrived damaged"),
            ("KB-DEL-02","my delivery contains a different item"),
        ],
    }

    channels = ["chat","email","app"]
    countries = ["IN","SG","GB","US"]
    priorities = ["low","medium","high"]
    products = ["mobile_app","credit_card","shopping","account_portal"]
    noise_prefix = ["","please help, ","urgent: ","hello team, ","need help - ","Hi, "]
    noise_suffix = [""," please"," asap","!!!"," thanks"," can you check?"," this is frustrating"]
    typo_map = {
        "charged":"chagred","refund":"refnd","password":"pasword","account":"acount",
        "delivery":"delivry","application":"applicaton","update":"udpate","payment":"paymnt"
    }

    def maybe_typo(text):
        if rng.random() < 0.12:
            candidates = [w for w in typo_map if w in text]
            if candidates:
                word = rng.choice(candidates)
                text = text.replace(word, typo_map[word], 1)
        return text

    def add_entity(text, intent):
        amount = order_id = email = event_date = ""
        if intent in ("billing","refund") and rng.random() < 0.65:
            amount = rng.choice(["₹499","₹1,299","₹4,299","$49.99","£25"])
            text += f" Amount {amount}."
        if intent == "delivery" and rng.random() < 0.75:
            order_id = f"ORD-{rng.randint(10000,99999)}"
            text += f" Order {order_id}."
        if intent in ("access","account") and rng.random() < 0.45:
            email = f"user{rng.randint(10,99)}@example.com"
            text += f" Email {email}."
        if rng.random() < 0.20:
            event_date = rng.choice(["12 Sep 2026","18/09/2026","Sep 20, 2026"])
            text += f" Date {event_date}."
        return text, amount, order_id, email, event_date

    target_counts = {
        "billing":90, "refund":80, "access":85,
        "technical":75, "account":65, "delivery":70
    }

    rows = []
    ticket_num = 1
    for intent, n in target_counts.items():
        for _ in range(n):
            article, base = rng.choice(templates[intent])
            text = rng.choice(noise_prefix) + base + rng.choice(noise_suffix)
            text = maybe_typo(text)
            text, amount, order_id, email, event_date = add_entity(text, intent)

            secondary = ""
            if rng.random() < 0.10:
                secondary_intent = rng.choice([x for x in templates if x != intent])
                _, secondary_text = rng.choice(templates[secondary_intent])
                secondary = secondary_intent
                text += " Also, " + secondary_text + "."

            rows.append({
                "ticket_id": f"TKT-{ticket_num:05d}",
                "text": text,
                "intent": intent,
                "kb_article_id": article,
                "channel": rng.choice(channels),
                "priority": rng.choices(priorities, weights=[0.45,0.40,0.15])[0],
                "country": rng.choice(countries),
                "product": rng.choice(products),
                "created_at": str(pd.Timestamp("2026-01-01") + pd.Timedelta(days=rng.randint(0,250)))[:10],
                "amount": amount,
                "order_id": order_id,
                "email": email,
                "event_date": event_date,
                "secondary_intent": secondary,
            })
            ticket_num += 1

    tickets = pd.DataFrame(rows)

    duplicates = tickets.sample(8, random_state=7).copy()
    duplicates["ticket_id"] = [f"TKT-{ticket_num+i:05d}" for i in range(len(duplicates))]
    tickets = pd.concat([tickets, duplicates], ignore_index=True)

    tickets.to_csv(DATA_RAW / "support_tickets.csv", index=False)
    kb.to_csv(DATA_RAW / "knowledge_base.csv", index=False)
    return tickets, kb

generated_tickets, generated_kb = build_educational_dataset(RANDOM_SEED)

print("Generated raw tickets:", len(generated_tickets))
print("Generated KB articles:", len(generated_kb))
print("Intent counts:", generated_tickets["intent"].value_counts().to_dict())

Generated raw tickets: 473
Generated KB articles: 12
Intent counts: {'billing': 92, 'access': 85, 'refund': 81, 'technical': 77, 'delivery': 71, 'account': 67}


## 3. Load and inspect the materialized raw data

The notebook writes ordinary CSV files so you can open and inspect every record outside Python.

In [3]:
tickets_raw = pd.read_csv(DATA_RAW / "support_tickets.csv")
kb = pd.read_csv(DATA_RAW / "knowledge_base.csv")

print("Tickets shape:", tickets_raw.shape)
print("Knowledge base shape:", kb.shape)
display(tickets_raw.head(5))
display(kb.head(5))

Tickets shape: (473, 14)
Knowledge base shape: (12, 4)


Example ticket columns:
ticket_id, text, intent, kb_article_id, channel, priority, country, product,
created_at, amount, order_id, email, event_date, secondary_intent

First intents: billing, billing, billing, billing, billing

Knowledge base includes 12 articles across billing, refund, access,
technical, account and delivery.

## 4. Data dictionary and structural validation

The target is `intent`. The retrieval target is `kb_article_id`. Entity ground-truth fields are `amount`, `order_id`, `email`, and `event_date`.

Before modeling, validate required fields, IDs, label domain, text presence and foreign-key references.

In [4]:
schema = pd.DataFrame({
    "column": tickets_raw.columns,
    "dtype": [str(tickets_raw[c].dtype) for c in tickets_raw.columns],
    "missing": [int(tickets_raw[c].isna().sum()) for c in tickets_raw.columns],
    "unique": [int(tickets_raw[c].nunique(dropna=True)) for c in tickets_raw.columns],
})
display(schema)

EXPECTED_INTENTS = {"billing","refund","access","technical","account","delivery"}
REQUIRED_COLUMNS = {
    "ticket_id","text","intent","kb_article_id","channel","priority",
    "country","product","created_at","amount","order_id","email","event_date"
}
quality_checks = {
    "required_columns_present": REQUIRED_COLUMNS.issubset(tickets_raw.columns),
    "ticket_id_unique": tickets_raw["ticket_id"].is_unique,
    "no_missing_text": tickets_raw["text"].notna().all(),
    "no_empty_text": tickets_raw["text"].fillna("").str.strip().ne("").all(),
    "intent_domain_valid": set(tickets_raw["intent"]) == EXPECTED_INTENTS,
    "kb_references_valid": set(tickets_raw["kb_article_id"]).issubset(set(kb["article_id"])),
    "kb_article_id_unique": kb["article_id"].is_unique,
}
quality_df = pd.DataFrame([{"check":k,"passed":bool(v)} for k,v in quality_checks.items()])
display(quality_df)
assert quality_df["passed"].all()
print("All structural data-contract checks passed.")

Structural checks:
required_columns_present  True
ticket_id_unique          True
no_missing_text           True
no_empty_text             True
intent_domain_valid       True
kb_references_valid       True
kb_article_id_unique      True

All structural data-contract checks passed.


## 5. EDA — class balance and operational metadata

EDA should drive modeling decisions. We inspect label balance and channel mix first, because metadata can become a spurious shortcut even if it is not intentionally used as a feature.

In [5]:
intent_counts = tickets_raw["intent"].value_counts().sort_index()
display(pd.concat([
    intent_counts.rename("count"),
    (intent_counts/len(tickets_raw)).rename("share")
],axis=1).round(3))

fig, ax = plt.subplots(figsize=(8,4))
intent_counts.sort_values(ascending=False).plot(kind="bar",ax=ax)
ax.set_title("Raw ticket count by intent")
ax.set_xlabel("Intent"); ax.set_ylabel("Tickets")
plt.tight_layout()
fig.savefig(ARTIFACTS/"eda_class_distribution.png",dpi=130)
fig.savefig(ARTIFACTS/"eda_class_distribution.svg",dpi=130)
plt.show()

           count  share
access        85  0.180
account       67  0.142
billing       92  0.194
delivery      71  0.150
refund        81  0.171
technical     77  0.163

**Rendered result**

![Raw ticket count by intent](artifacts/eda_class_distribution.svg)

In [6]:
channel_intent = pd.crosstab(
    tickets_raw["intent"], tickets_raw["channel"], normalize="index"
).round(3)
display(channel_intent)

fig, ax = plt.subplots(figsize=(7,4))
im=ax.imshow(channel_intent.values,aspect="auto",vmin=0,vmax=1)
ax.set_xticks(range(len(channel_intent.columns)),channel_intent.columns)
ax.set_yticks(range(len(channel_intent.index)),channel_intent.index)
ax.set_title("Channel share within each intent")
for i in range(channel_intent.shape[0]):
    for j in range(channel_intent.shape[1]):
        ax.text(j,i,f"{channel_intent.iloc[i,j]:.2f}",ha="center",va="center")
fig.colorbar(im,ax=ax,label="share")
plt.tight_layout()
fig.savefig(ARTIFACTS/"eda_channel_intent.png",dpi=130)
fig.savefig(ARTIFACTS/"eda_channel_intent.svg",dpi=130)
plt.show()

channel        app   chat  email
intent
access       0.247  0.341  0.412
account      0.388  0.269  0.343
billing      0.435  0.304  0.261
delivery     0.352  0.366  0.282
refund       0.370  0.358  0.272
technical    0.364  0.312  0.325

**Rendered result**

![Channel share](artifacts/eda_channel_intent.svg)

## 6. Text EDA — length, duplicates, noise, vocabulary signals

Text systems are sensitive to sequence length, duplicated examples, PII-like tokens and ambiguity. These are data-quality questions, not cosmetic visualization.

In [7]:
eda=tickets_raw.copy()
eda["char_len"]=eda["text"].str.len()
eda["token_len"]=eda["text"].str.findall(r"\b\w+\b").str.len()
eda["has_email_text"]=eda["text"].str.contains(r"[\w.+-]+@[\w.-]+\.\w+",regex=True)
eda["has_amount_text"]=eda["text"].str.contains(r"(?:₹|\$|£)\s?\d",regex=True)
eda["has_order_text"]=eda["text"].str.contains(r"ORD-\d{5}",regex=True)
eda["has_secondary_intent"]=eda["secondary_intent"].fillna("").ne("")

display(eda[["char_len","token_len"]].describe(percentiles=[.1,.5,.9,.95,.99]).round(2))
display(pd.Series({
    "exact_duplicate_text_rows":int(eda.duplicated("text",keep=False).sum()),
    "unique_texts":int(eda["text"].nunique()),
    "contains_email":int(eda["has_email_text"].sum()),
    "contains_amount":int(eda["has_amount_text"].sum()),
    "contains_order_id":int(eda["has_order_text"].sum()),
    "controlled_multi_intent":int(eda["has_secondary_intent"].sum()),
}).to_frame("count"))

fig, ax=plt.subplots(figsize=(8,4))
for intent,g in eda.groupby("intent"):
    ax.hist(g["token_len"],bins=18,alpha=.35,label=intent)
ax.set_title("Token-length distribution by intent")
ax.set_xlabel("Tokens"); ax.set_ylabel("Count")
ax.legend(ncol=3,fontsize=8)
plt.tight_layout()
fig.savefig(ARTIFACTS/"eda_token_length.png",dpi=130)
fig.savefig(ARTIFACTS/"eda_token_length.svg",dpi=130)
plt.show()

Text-length and noise audit completed.
Raw rows: 473
The corpus contains exact duplicates, entity-bearing rows, typos and controlled multi-intent examples.

**Rendered result**

![Token length](artifacts/eda_token_length.svg)

## 7. Lexical EDA and EDA → modeling decisions

Class-specific n-grams tell us which local phrases carry signal. The final classifier will use **text only**; operational metadata is retained for slicing, not fed into the model by default.

Decisions:
1. class imbalance → select with macro-F1;
2. typos → compare a character n-gram representation;
3. duplicates → remove before splitting;
4. multi-intent examples → expect useful errors rather than synthetic perfection;
5. entities → extract separately on raw text.

In [8]:
def top_ngrams(texts,ngram_range=(2,2),top_n=8):
    vec=CountVectorizer(ngram_range=ngram_range,stop_words="english",min_df=2)
    X=vec.fit_transform(texts)
    counts=np.asarray(X.sum(axis=0)).ravel()
    terms=np.asarray(vec.get_feature_names_out())
    order=counts.argsort()[::-1][:top_n]
    return list(zip(terms[order],counts[order]))

rows=[]
for intent,g in tickets_raw.groupby("intent"):
    for term,count in top_ngrams(g["text"]):
        rows.append({"intent":intent,"bigram":term,"count":int(count)})
top_bigram_df=pd.DataFrame(rows)
display(top_bigram_df.groupby("intent").head(5).reset_index(drop=True))

Class-specific bigrams were computed for all six intents.
Examples include password/account-lock phrases for access,
refund phrases for refund, and delivery/order phrases for delivery.

## 8. Cleaning / normalization as a versioned contract

Cleaning must preserve business evidence. The classifier replaces high-cardinality operational entities with semantic placeholders, while entity extraction later runs on the raw text.

In [9]:
EMAIL_RE = re.compile(r"[\w.+-]+@[\w.-]+\.\w+")
ORDER_RE = re.compile(r"\bORD-\d{5}\b",re.I)
AMOUNT_RE = re.compile(r"(?:₹|\$|£)\s?\d[\d,]*(?:\.\d{1,2})?")
DATE_RE = re.compile(
    r"\b(?:\d{1,2}\s+[A-Za-z]{3,9}\s+\d{4}|"
    r"\d{1,2}/\d{1,2}/\d{4}|"
    r"[A-Za-z]{3,9}\s+\d{1,2},\s*\d{4})\b"
)

def normalize_text(text):
    text=unicodedata.normalize("NFKC",str(text))
    text=EMAIL_RE.sub(" <EMAIL> ",text)
    text=ORDER_RE.sub(" <ORDER_ID> ",text)
    text=AMOUNT_RE.sub(" <AMOUNT> ",text)
    text=DATE_RE.sub(" <DATE> ",text)
    return re.sub(r"\s+"," ",text).strip()

examples=tickets_raw.sample(6,random_state=42)[["ticket_id","text"]].copy()
examples["normalized"]=examples["text"].map(normalize_text)
display(examples)
assert normalize_text(normalize_text("  hello   user1@example.com ")) == normalize_text("  hello   user1@example.com ")
print("Normalization idempotence check passed.")

Normalization idempotence check passed.


## 9. Duplicate handling before splitting

Removing duplicates **after** splitting can leak near-identical examples into evaluation partitions. We deduplicate normalized text before creating train/validation/test sets.

In [10]:
tickets=tickets_raw.copy()
tickets["normalized_text"]=tickets["text"].map(normalize_text)

before=len(tickets)
duplicate_mask=tickets.duplicated("normalized_text",keep="first")
tickets=tickets.loc[~duplicate_mask].reset_index(drop=True)
removed=before-len(tickets)

tickets.to_csv(DATA_INTERIM/"cleaned_tickets.csv",index=False)
print("Rows before deduplication:",before)
print("Rows after deduplication :",len(tickets))
print("Duplicates removed       :",removed)

Rows before deduplication: 473
Rows after deduplication : 445
Duplicates removed       : 28


## 10. Stratified train / validation / test

- **60% train** → learn vectorizer vocabulary and model parameters
- **20% validation** → compare candidates and tune `C`
- **20% test** → one final evaluation after selection

The test set is not consulted during selection.

In [11]:
trainval,test=train_test_split(
    tickets,test_size=0.20,random_state=RANDOM_SEED,stratify=tickets["intent"]
)
train,val=train_test_split(
    trainval,test_size=0.25,random_state=RANDOM_SEED,stratify=trainval["intent"]
)

for name,part in [("train",train),("validation",val),("test",test)]:
    part.to_csv(DATA_PROCESSED/f"{name}.csv",index=False)

split_summary=pd.concat({
    "train":train["intent"].value_counts(normalize=True),
    "validation":val["intent"].value_counts(normalize=True),
    "test":test["intent"].value_counts(normalize=True),
},axis=1).fillna(0).round(3)
display(split_summary)
print("Split sizes:",{"train":len(train),"validation":len(val),"test":len(test)})
assert set(train.ticket_id).isdisjoint(val.ticket_id)
assert set(train.ticket_id).isdisjoint(test.ticket_id)
assert set(val.ticket_id).isdisjoint(test.ticket_id)
print("ID overlap check passed.")

Split sizes: {'train': 267, 'validation': 89, 'test': 89}
ID overlap check passed.


## 11. Baselines and candidate representations

We compare:
1. majority baseline;
2. Bag-of-Words + Multinomial Naive Bayes;
3. word TF-IDF + Logistic Regression;
4. word + character TF-IDF + Logistic Regression.

All vectorizers live **inside** their pipelines, preventing preprocessing leakage.

In [12]:
X_train,y_train=train["normalized_text"],train["intent"]
X_val,y_val=val["normalized_text"],val["intent"]

def make_candidates():
    return {
        "majority":Pipeline([
            ("features",CountVectorizer()),
            ("model",DummyClassifier(strategy="most_frequent"))
        ]),
        "bow_nb":Pipeline([
            ("features",CountVectorizer(ngram_range=(1,2),min_df=2)),
            ("model",MultinomialNB(alpha=0.6))
        ]),
        "word_tfidf_logreg":Pipeline([
            ("features",TfidfVectorizer(
                ngram_range=(1,2),min_df=2,sublinear_tf=True,strip_accents="unicode"
            )),
            ("model",LogisticRegression(
                max_iter=1500,class_weight="balanced",random_state=RANDOM_SEED
            ))
        ]),
        "hybrid_tfidf_logreg":Pipeline([
            ("features",FeatureUnion([
                ("word",TfidfVectorizer(
                    ngram_range=(1,2),min_df=2,sublinear_tf=True,strip_accents="unicode"
                )),
                ("char",TfidfVectorizer(
                    analyzer="char_wb",ngram_range=(3,5),min_df=2,sublinear_tf=True
                ))
            ])),
            ("model",LogisticRegression(
                max_iter=1500,class_weight="balanced",random_state=RANDOM_SEED
            ))
        ])
    }

results=[]
fitted_candidates={}
for name,model in make_candidates().items():
    model.fit(X_train,y_train)
    pred=model.predict(X_val)
    results.append({
        "model":name,
        "validation_accuracy":accuracy_score(y_val,pred),
        "validation_macro_f1":f1_score(y_val,pred,average="macro"),
        "validation_weighted_f1":f1_score(y_val,pred,average="weighted"),
    })
    fitted_candidates[name]=model

comparison=pd.DataFrame(results).sort_values(
    ["validation_macro_f1","validation_accuracy"],ascending=False
).reset_index(drop=True)
display(comparison.round(4))
comparison.to_csv(ARTIFACTS/"model_comparison.csv",index=False)

fig,ax=plt.subplots(figsize=(8,4))
plot_df=comparison.sort_values("validation_macro_f1")
ax.barh(plot_df["model"],plot_df["validation_macro_f1"])
ax.set_xlim(0,1.05); ax.set_xlabel("Validation macro-F1")
ax.set_title("Baseline and candidate model comparison")
plt.tight_layout()
fig.savefig(ARTIFACTS/"model_comparison.png",dpi=130)
fig.savefig(ARTIFACTS/"model_comparison.svg",dpi=130)
plt.show()

                 model  validation_accuracy  validation_macro_f1
0               bow_nb               0.9888               0.9877
1  hybrid_tfidf_logreg               0.9888               0.9877
2    word_tfidf_logreg               0.9775               0.9768
3             majority               0.2022               0.0561

**Rendered result**

![Model comparison](artifacts/model_comparison.svg)

## 12. Validation-only hyperparameter tuning

The strongest production-friendly representation is tuned on the validation set only. We search a small explicit set of logistic-regression regularization strengths for transparency.

In [13]:
base_name="hybrid_tfidf_logreg"
C_values=[0.25,0.5,1.0,2.0,4.0]
tuning_rows=[]
best_score=-1
best_C=None

for C in C_values:
    model=make_candidates()[base_name]
    model.set_params(model__C=C)
    model.fit(X_train,y_train)
    pred=model.predict(X_val)
    score=f1_score(y_val,pred,average="macro")
    tuning_rows.append({"C":C,"validation_macro_f1":score})
    if score>best_score:
        best_score=score; best_C=C

tuning=pd.DataFrame(tuning_rows)
display(tuning.round(4))
print("Selected C:",best_C)
print("Best validation macro-F1:",round(best_score,4))
tuning.to_csv(ARTIFACTS/"hyperparameter_tuning.csv",index=False)

      C  validation_macro_f1
0  0.25               0.9877
1  0.50               0.9877
2  1.00               0.9877
3  2.00               0.9877
4  4.00               0.9877

Selected C: 0.25
Best validation macro-F1: 0.9877


## 13. Refit on train + validation, then evaluate once on test

Only after selection do we combine train and validation. The untouched test set now estimates final generalization for this educational experiment.

In [14]:
trainval_final=pd.concat([train,val],ignore_index=True)
final_model=make_candidates()[base_name]
final_model.set_params(model__C=best_C)
final_model.fit(trainval_final["normalized_text"],trainval_final["intent"])

X_test=test["normalized_text"]; y_test=test["intent"]
test_pred=final_model.predict(X_test)
test_proba=final_model.predict_proba(X_test)
classes=final_model.named_steps["model"].classes_

test_accuracy=accuracy_score(y_test,test_pred)
test_macro_f1=f1_score(y_test,test_pred,average="macro")
test_weighted_f1=f1_score(y_test,test_pred,average="weighted")

print("FINAL UNTOUCHED TEST METRICS")
print("accuracy    :",round(test_accuracy,4))
print("macro-F1   :",round(test_macro_f1,4))
print("weighted-F1:",round(test_weighted_f1,4))
print()
print(classification_report(y_test,test_pred,digits=3,zero_division=0))

FINAL UNTOUCHED TEST METRICS
accuracy    : 0.8989
macro-F1   : 0.8984
weighted-F1: 0.8999

              precision    recall  f1-score   support
      access      0.933     0.875     0.903        16
     account      0.909     0.833     0.870        12
     billing      0.941     0.889     0.914        18
    delivery      1.000     1.000     1.000        14
      refund      0.875     0.933     0.903        15
   technical      0.750     0.857     0.800        14

    accuracy                          0.899        89
   macro avg      0.901     0.898     0.898        89
weighted avg      0.903     0.899     0.900        89


In [15]:
report=pd.DataFrame(
    classification_report(y_test,test_pred,output_dict=True,zero_division=0)
).T
report.to_csv(ARTIFACTS/"classification_report.csv")

labels=sorted(tickets["intent"].unique())
cm=confusion_matrix(y_test,test_pred,labels=labels)

fig,ax=plt.subplots(figsize=(7,6))
im=ax.imshow(cm)
ax.set_xticks(range(len(labels)),labels,rotation=45,ha="right")
ax.set_yticks(range(len(labels)),labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Final test confusion matrix")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j,i,str(cm[i,j]),ha="center",va="center")
fig.colorbar(im,ax=ax)
plt.tight_layout()
fig.savefig(ARTIFACTS/"confusion_matrix.png",dpi=140)
fig.savefig(ARTIFACTS/"confusion_matrix.svg",dpi=140)
plt.show()

Confusion matrix labels:
['access', 'account', 'billing', 'delivery', 'refund', 'technical']

[[14, 1, 0, 0, 0, 1],
 [ 1,10, 0, 0, 1, 0],
 [ 0, 0,16, 0, 0, 2],
 [ 0, 0, 0,14, 0, 0],
 [ 0, 0, 0, 0,14, 1],
 [ 0, 0, 1, 0, 1,12]]

**Rendered result**

![Confusion matrix](artifacts/confusion_matrix.svg)

## 14. Confidence, abstention and coverage

Production systems do not need to automate every case. We inspect maximum predicted probability as a simple confidence signal and compare automation coverage against accepted-case accuracy.

Raw classifier probabilities are not assumed to be perfectly calibrated; a real deployment should validate/calibrate them on sufficient held-out data.

In [16]:
confidence=test_proba.max(axis=1)
pred_df=test[["ticket_id","text","intent","secondary_intent"]].copy()
pred_df["prediction"]=test_pred
pred_df["confidence"]=confidence
pred_df["correct"]=pred_df["intent"].eq(pred_df["prediction"])

threshold_rows=[]
for threshold in [0.40,0.50,0.60,0.70,0.80]:
    accepted=pred_df["confidence"]>=threshold
    coverage=accepted.mean()
    accepted_acc=pred_df.loc[accepted,"correct"].mean() if accepted.any() else np.nan
    threshold_rows.append({
        "threshold":threshold,
        "coverage":coverage,
        "accepted_accuracy":accepted_acc,
        "human_review_rate":1-coverage
    })
threshold_df=pd.DataFrame(threshold_rows)
display(threshold_df.round(3))

fig,ax=plt.subplots(figsize=(7,4))
ax.hist(pred_df.loc[pred_df.correct,"confidence"],bins=12,alpha=.65,label="correct")
ax.hist(pred_df.loc[~pred_df.correct,"confidence"],bins=12,alpha=.65,label="incorrect")
ax.set_xlabel("Maximum predicted probability")
ax.set_ylabel("Tickets")
ax.set_title("Confidence distribution by correctness")
ax.legend()
plt.tight_layout()
fig.savefig(ARTIFACTS/"confidence_distribution.png",dpi=130)
fig.savefig(ARTIFACTS/"confidence_distribution.svg",dpi=130)
plt.show()

   threshold  coverage  accepted_accuracy  human_review_rate
0        0.4     0.843              1.000              0.157
1        0.5     0.753              1.000              0.247
2        0.6     0.539              1.000              0.461
3        0.7     0.157              1.000              0.843
4        0.8     0.022              1.000              0.978

**Rendered result**

![Confidence distribution](artifacts/confidence_distribution.svg)

## 15. Error analysis

A score does not tell us what to fix. We store wrong predictions and categorize likely failure modes such as multi-intent ambiguity, low confidence and label overlap.

In [17]:
pred_df["token_len"]=pred_df["text"].str.findall(r"\b\w+\b").str.len()
errors=pred_df.loc[~pred_df["correct"]].copy()

def error_category(row):
    if pd.notna(row["secondary_intent"]) and str(row["secondary_intent"]).strip():
        return "multi_intent"
    if row["confidence"]<0.60:
        return "low_confidence_ambiguity"
    if row["token_len"]<=6:
        return "short_text"
    return "lexical_or_label_overlap"

if len(errors):
    errors["error_category"]=errors.apply(error_category,axis=1)
    display(errors[[
        "ticket_id","text","intent","prediction","confidence",
        "secondary_intent","error_category"
    ]].sort_values("confidence"))
    display(errors["error_category"].value_counts().to_frame("count"))

errors.to_csv(ARTIFACTS/"errors.csv",index=False)
pred_df.to_csv(ARTIFACTS/"test_predictions.csv",index=False)
print("Test errors:",len(errors))

Test errors: 9


## 16. Inspect learned linear features

Linear coefficients are a useful **model-behavior diagnostic**. They show which features increase each class logit, but they should not be confused with causal explanations of language.

In [18]:
feature_union=final_model.named_steps["features"]
feature_names=np.array(feature_union.get_feature_names_out())
coef=final_model.named_steps["model"].coef_

top_features={}
for i,label in enumerate(classes):
    idx=np.argsort(coef[i])[-10:][::-1]
    top_features[label]=feature_names[idx].tolist()

top_feature_df=pd.DataFrame(top_features)
display(top_feature_df)
top_feature_df.to_csv(ARTIFACTS/"top_features_by_class.csv",index=False)

Top positive features were extracted for each of:
access, account, billing, delivery, refund, technical.
The table is also written to artifacts/top_features_by_class.csv.

## 17. Entity extraction on raw text

The classifier uses normalized placeholders, but operational extraction runs on **raw text**. We evaluate exact-value extraction against committed/generator ground truth for amount, order ID, email and event date.

In [19]:
def extract_entities(text):
    text=str(text)
    def one(pattern):
        m=pattern.search(text)
        return m.group(0) if m else ""
    return {
        "amount":one(AMOUNT_RE),
        "order_id":one(ORDER_RE),
        "email":one(EMAIL_RE),
        "event_date":one(DATE_RE),
    }

entity_metrics=[]
for field in ["amount","order_id","email","event_date"]:
    gold=tickets_raw[field].fillna("").astype(str)
    pred=tickets_raw["text"].map(lambda x: extract_entities(x)[field])
    gold_pos=gold.ne(""); pred_pos=pred.ne("")
    tp=((gold==pred)&gold_pos).sum()
    fp=(pred_pos&(pred!=gold)).sum()
    fn=(gold_pos&(pred!=gold)).sum()
    precision=tp/(tp+fp) if tp+fp else 1.0
    recall=tp/(tp+fn) if tp+fn else 1.0
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
    entity_metrics.append({
        "entity":field,"tp":int(tp),"fp":int(fp),"fn":int(fn),
        "precision":precision,"recall":recall,"f1":f1
    })

entity_metrics_df=pd.DataFrame(entity_metrics)
display(entity_metrics_df.round(3))
entity_metrics_df.to_csv(ARTIFACTS/"entity_extraction_metrics.csv",index=False)

       entity   tp  fp  fn  precision  recall   f1
0      amount  109   0   0        1.0     1.0  1.0
1    order_id   57   0   0        1.0     1.0  1.0
2       email   62   0   0        1.0     1.0  1.0
3  event_date   93   0   0        1.0     1.0  1.0

The perfect entity score is intentional for this structured educational schema. It demonstrates how to **measure** extraction; it must not be interpreted as evidence that regex solves real-world NER.

## 18. Knowledge-base retrieval

Each ticket has a relevant support article ID. We compare:
- word TF-IDF cosine retrieval;
- LSA dense retrieval (TF-IDF → SVD → cosine).

This is evaluated as a ranking problem with Recall@k and MRR.

In [20]:
kb=kb.copy()
kb["document"]=kb["title"].fillna("")+". "+kb["content"].fillna("")

word_retriever=TfidfVectorizer(
    ngram_range=(1,2),stop_words="english",sublinear_tf=True
)
kb_word=word_retriever.fit_transform(kb["document"])

lsa_tfidf=TfidfVectorizer(
    ngram_range=(1,2),stop_words="english",sublinear_tf=True
)
kb_sparse=lsa_tfidf.fit_transform(kb["document"])
n_components=min(8,kb_sparse.shape[0]-1,kb_sparse.shape[1]-1)
lsa=TruncatedSVD(n_components=n_components,random_state=RANDOM_SEED)
kb_lsa=Normalizer().fit_transform(lsa.fit_transform(kb_sparse))

def rank_word(query):
    q=word_retriever.transform([normalize_text(query)])
    scores=cosine_similarity(q,kb_word).ravel()
    return np.argsort(-scores),scores

def rank_lsa(query):
    q=lsa_tfidf.transform([normalize_text(query)])
    q_dense=Normalizer().fit_transform(lsa.transform(q))
    scores=cosine_similarity(q_dense,kb_lsa).ravel()
    return np.argsort(-scores),scores

def retrieval_metrics(frame,ranker):
    rr=[]; hit1=[]; hit3=[]; details=[]
    for _,row in frame.iterrows():
        order,scores=ranker(row["text"])
        ranked_ids=kb.iloc[order]["article_id"].tolist()
        target=row["kb_article_id"]
        rank=ranked_ids.index(target)+1
        rr.append(1/rank); hit1.append(rank<=1); hit3.append(rank<=3)
        details.append({
            "ticket_id":row["ticket_id"],"relevant_article":target,
            "top1":ranked_ids[0],"rank":rank,"top1_score":float(scores[order[0]])
        })
    return {
        "Recall@1":float(np.mean(hit1)),
        "Recall@3":float(np.mean(hit3)),
        "MRR":float(np.mean(rr))
    },pd.DataFrame(details)

word_metrics,word_details=retrieval_metrics(test,rank_word)
lsa_metrics,lsa_details=retrieval_metrics(test,rank_lsa)

retrieval_comparison=pd.DataFrame(
    [word_metrics,lsa_metrics],index=["word_tfidf","lsa_dense"]
)
display(retrieval_comparison.round(3))
retrieval_comparison.to_csv(ARTIFACTS/"retrieval_metrics.csv")
word_details.to_csv(ARTIFACTS/"retrieval_predictions.csv",index=False)

fig,ax=plt.subplots(figsize=(7,4))
retrieval_comparison.T.plot(kind="bar",ax=ax)
ax.set_ylim(0,1.05); ax.set_ylabel("Score")
ax.set_title("Retrieval evaluation on held-out test tickets")
plt.tight_layout()
fig.savefig(ARTIFACTS/"retrieval_metrics.png",dpi=130)
fig.savefig(ARTIFACTS/"retrieval_metrics.svg",dpi=130)
plt.show()

            Recall@1  Recall@3    MRR
word_tfidf     0.640      0.854  0.762
lsa_dense      0.652      0.843  0.767

**Rendered result**

![Retrieval metrics](artifacts/retrieval_metrics.svg)

## 19. Integrated end-to-end inference

The deployed interface composes three specialized paths:
- normalized text → intent classifier;
- raw text → entity extractor;
- raw text → knowledge retriever.

A confidence threshold controls human review.

In [21]:
PRODUCTION_THRESHOLD=0.60

def retrieve_article(query):
    order,scores=rank_word(query)
    i=int(order[0])
    return {
        "article_id":kb.iloc[i]["article_id"],
        "title":kb.iloc[i]["title"],
        "score":float(scores[i]),
    }

def predict_ticket(text,model=final_model,threshold=PRODUCTION_THRESHOLD):
    raw=str(text).strip()
    if not raw:
        raise ValueError("text must be a non-empty string")
    normalized=normalize_text(raw)
    proba=model.predict_proba([normalized])[0]
    classes_local=model.named_steps["model"].classes_
    idx=int(np.argmax(proba))
    intent=str(classes_local[idx])
    confidence=float(proba[idx])
    entities={k:v for k,v in extract_entities(raw).items() if v}
    article=retrieve_article(raw)
    return {
        "intent":intent,
        "confidence":round(confidence,4),
        "entities":entities,
        "recommended_article_id":article["article_id"],
        "recommended_article_title":article["title"],
        "retrieval_score":round(article["score"],4),
        "requires_human_review":bool(confidence<threshold),
    }

examples=[
    "My card was charged twice for ₹4,299 on 12 Sep 2026.",
    "I forgot my password and OTP is not arriving. Email user42@example.com",
    "Order ORD-45678 arrived damaged.",
]
for x in examples:
    print("\nINPUT:",x)
    print(json.dumps(predict_ticket(x),indent=2,ensure_ascii=False))

INPUT: My card was charged twice for ₹4,299 on 12 Sep 2026.
{
  "intent": "billing",
  "entities": {"amount": "₹4,299", "event_date": "12 Sep 2026"},
  "recommended_article_id": "KB-BILL-01"
}

INPUT: I forgot my password and OTP is not arriving. Email user42@example.com
{
  "intent": "access",
  "entities": {"email": "user42@example.com"},
  "recommended_article_id": "KB-ACC-02"
}

INPUT: Order ORD-45678 arrived damaged.
{
  "intent": "delivery",
  "entities": {"order_id": "ORD-45678"},
  "recommended_article_id": "KB-DEL-02"
}


## 20. Serialize the full classifier pipeline and metadata

The fitted TF-IDF feature union and classifier are stored **together**. This is essential: saving only the estimator would create train/inference preprocessing skew.

In [22]:
MODEL_PATH=ARTIFACTS/"intent_classifier.joblib"
dump(final_model,MODEL_PATH)

metrics_payload={
    "random_seed":RANDOM_SEED,
    "selected_representation":base_name,
    "selected_C":best_C,
    "validation_macro_f1":float(best_score),
    "test_accuracy":float(test_accuracy),
    "test_macro_f1":float(test_macro_f1),
    "test_weighted_f1":float(test_weighted_f1),
    "production_threshold":PRODUCTION_THRESHOLD,
    "classes":classes.tolist(),
}
(ARTIFACTS/"metrics.json").write_text(json.dumps(metrics_payload,indent=2))

metadata={
    "project":"customer_support_intelligence",
    "model_type":"word + character TF-IDF + LogisticRegression",
    "python":platform.python_version(),
    "scikit_learn":sklearn_version,
    "training_rows":int(len(trainval_final)),
    "test_rows":int(len(test)),
    "raw_dataset_rows":int(len(tickets_raw)),
    "deduplicated_rows":int(len(tickets)),
    "input_field":"text",
    "output_intents":classes.tolist(),
}
(ARTIFACTS/"model_metadata.json").write_text(json.dumps(metadata,indent=2))

print(json.dumps(metrics_payload,indent=2))

{
  "random_seed": 42,
  "selected_representation": "hybrid_tfidf_logreg",
  "selected_C": 0.25,
  "validation_macro_f1": 0.9877031181379007,
  "test_accuracy": 0.898876404494382,
  "test_macro_f1": 0.8983837574300407,
  "test_weighted_f1": 0.899909274897062,
  "production_threshold": 0.6
}


## 21. Reload verification

A persisted artifact is not trusted until it is loaded back and produces numerically identical predictions.

In [23]:
reloaded_model=load(MODEL_PATH)

roundtrip_text="My refund of $49.99 is still missing."
before=final_model.predict_proba([normalize_text(roundtrip_text)])
after=reloaded_model.predict_proba([normalize_text(roundtrip_text)])

assert np.allclose(before,after)
print("Serialization round-trip check passed.")
print(json.dumps(
    predict_ticket(roundtrip_text,model=reloaded_model),
    indent=2
))

Serialization round-trip check passed.
{
  "intent": "refund",
  "confidence": 0.6667,
  "entities": {"amount": "$49.99"},
  "recommended_article_id": "KB-REF-02",
  "recommended_article_title": "Refund amount incorrect",
  "retrieval_score": 0.2775,
  "requires_human_review": false
}


## 22. Robustness / edge-case regression tests

We test casing, typo noise, whitespace, emoji, IDs and PII-like text. Production robustness should be expanded with domain-specific adversarial and out-of-distribution cases.

In [24]:
robustness_cases=[
    "GREAT!!! my CARD was CHARGED twice!!!",
    "my card was chagred twice",
    "please   reset      my password",
    "Order ORD-99999 is late 😡",
    "user77@example.com cannot receive the OTP",
]
rows=[]
for text in robustness_cases:
    out=predict_ticket(text,model=reloaded_model)
    rows.append({
        "text":text,
        "intent":out["intent"],
        "confidence":out["confidence"],
        "article":out["recommended_article_id"],
        "human_review":out["requires_human_review"],
    })
display(pd.DataFrame(rows))

assert normalize_text(" x  ")=="x"
assert extract_entities("Pay ₹499 for ORD-12345")["amount"]=="₹499"
assert extract_entities("Pay ₹499 for ORD-12345")["order_id"].upper()=="ORD-12345"
print("Robustness/contract assertions passed.")

                                        text     intent  confidence      article  human_review
0      GREAT!!! my CARD was CHARGED twice!!!    billing      0.5329   KB-BILL-01          True
1                  my card was chagred twice    billing      0.4108   KB-BILL-01          True
2            please   reset      my password     access      0.4675    KB-ACC-01          True
3                  Order ORD-99999 is late 😡  delivery      0.7322    KB-DEL-01         False
4  user77@example.com cannot receive the OTP     access      0.3966    KB-ACC-02          True

Robustness/contract assertions passed.


## 23. Monitoring simulation — input and prediction drift

We simulate later traffic with new terms such as *passkey*, *biometric* and *wallet*. We monitor:
- mean input length;
- production OOV rate relative to the trained word vocabulary;
- prediction-distribution divergence.

These are drift **signals**, not proof that concept drift has occurred.

In [25]:
prod_batch=[
    "My biometric passkey login stopped working after the security update and I cannot access the wallet",
    "The wallet payment is duplicated and the merchant says one entry should disappear",
    "Package tracking has not moved for five days and the courier chatbot cannot locate my parcel",
    "Please update my recovery email and mobile number because the old contact details are no longer valid",
    "The application freezes after biometric verification and then shows a gateway timeout",
    "My refund for the wallet transaction still has not arrived after ten business days",
]*12

train_lengths=trainval_final["text"].str.findall(r"\b\w+\b").str.len().to_numpy()
prod_lengths=np.array([len(re.findall(r"\b\w+\b",x)) for x in prod_batch])

word_vec=reloaded_model.named_steps["features"].transformer_list[0][1]
train_vocab=set(word_vec.vocabulary_.keys())
prod_tokens=[t.lower() for x in prod_batch for t in re.findall(r"\b\w+\b",x)]
oov_rate=np.mean([t not in train_vocab for t in prod_tokens])

train_pred=reloaded_model.predict(trainval_final["normalized_text"])
prod_pred=reloaded_model.predict([normalize_text(x) for x in prod_batch])

def distribution(values,labels):
    c=Counter(values)
    arr=np.array([c.get(x,0) for x in labels],dtype=float)
    return arr/arr.sum()

p=distribution(train_pred,classes)
q=distribution(prod_pred,classes)

def js_divergence(p,q):
    eps=1e-12
    p=np.clip(p,eps,1); q=np.clip(q,eps,1); m=.5*(p+q)
    return .5*np.sum(p*np.log(p/m))+.5*np.sum(q*np.log(q/m))

monitoring=pd.DataFrame({
    "metric":[
        "mean_token_length_train","mean_token_length_production",
        "production_OOV_rate","prediction_JS_divergence"
    ],
    "value":[train_lengths.mean(),prod_lengths.mean(),oov_rate,js_divergence(p,q)]
})
display(monitoring.round(4))
monitoring.to_csv(ARTIFACTS/"monitoring_snapshot.csv",index=False)

fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].hist(train_lengths,bins=15,alpha=.65,label="train")
axes[0].hist(prod_lengths,bins=15,alpha=.65,label="simulated production")
axes[0].set_title("Input-length drift"); axes[0].legend()

x=np.arange(len(classes))
axes[1].bar(x-.18,p,width=.36,label="train predictions")
axes[1].bar(x+.18,q,width=.36,label="production predictions")
axes[1].set_xticks(x,classes,rotation=45,ha="right")
axes[1].set_title("Prediction-distribution shift"); axes[1].legend()
plt.tight_layout()
fig.savefig(ARTIFACTS/"monitoring_drift.png",dpi=130)
fig.savefig(ARTIFACTS/"monitoring_drift.svg",dpi=130)
plt.show()

                         metric    value
0       mean_token_length_train  12.4592
1  mean_token_length_production  14.6667
2           production_OOV_rate   0.3977
3      prediction_JS_divergence   0.0020

**Rendered result**

![Monitoring drift](artifacts/monitoring_drift.svg)

## 24. Retraining and promotion policy

Retraining should be triggered by evidence, not merely by a calendar.

Example signals:
- delayed-label macro-F1 declines;
- high-confidence error rate rises;
- OOV/lexical drift remains elevated;
- business-real class-prior shift persists;
- human override rate rises;
- a new product/process changes language materially.

Promotion flow:

new labeled data → quality checks → retrain candidate → locked evaluation suite → compare with incumbent → robustness/regression tests → canary/shadow or reject.

## 25. Artifact manifest

The notebook writes raw, intermediate and processed data plus model/evaluation artifacts so nothing important exists only in notebook memory.

In [26]:
manifest={
    "raw_data":{
        "support_tickets":"data/raw/support_tickets.csv",
        "knowledge_base":"data/raw/knowledge_base.csv",
    },
    "intermediate":["data/interim/cleaned_tickets.csv"],
    "processed":[
        "data/processed/train.csv",
        "data/processed/validation.csv",
        "data/processed/test.csv",
    ],
    "artifacts":sorted([p.name for p in ARTIFACTS.iterdir() if p.is_file()]),
    "seed":RANDOM_SEED,
}
(ARTIFACTS/"run_manifest.json").write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))

Artifact manifest written.
Generated categories:
- raw educational CSVs
- cleaned/intermediate CSV
- train/validation/test CSVs
- serialized intent_classifier.joblib
- model comparison + tuning metrics
- final classification report + predictions + errors
- entity extraction metrics
- retrieval metrics + predictions
- monitoring snapshot
- model metadata + run manifest
- PNG/SVG figures


# Final end-to-end review

This single notebook has now demonstrated:

raw educational dataset → validation → EDA → leakage prevention → normalization → train/validation/test → majority baseline → BoW + Naive Bayes → word TF-IDF → word+character TF-IDF → validation tuning → final test → confusion matrix → confidence/abstention → error analysis → feature inspection → entity extraction → retrieval → integrated inference → serialization → reload verification → robustness → monitoring/drift → retraining policy.

### Reference run
- validation macro-F1: **0.9877**
- final test accuracy: **0.8989**
- final test macro-F1: **0.8984**
- final test weighted-F1: **0.8999**
- test errors available for analysis: **9**
- entity exact-match F1 in this controlled educational schema: **1.0**
- retrieval Word TF-IDF: Recall@1 **0.6404**, Recall@3 **0.8539**, MRR **0.7623**
- retrieval LSA: Recall@1 **0.6517**, Recall@3 **0.8427**, MRR **0.7670**

### Experiments to try next
1. change normalization rules;
2. remove character n-grams and inspect typo robustness;
3. alter the abstention threshold and compare coverage vs accepted-case accuracy;
4. introduce time-based splitting;
5. add a new intent and regenerate the dataset;
6. replace TF-IDF retrieval with a pretrained sentence transformer;
7. add a real calibration split and probability calibration;
8. add delayed labels and implement incumbent-vs-candidate promotion gates.

The key habit is:

**define the contract → inspect the data → establish a baseline → prevent leakage → evaluate honestly → inspect failures → package the full inference path → monitor the deployed system.**